In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import os
pd.options.mode.chained_assignment = None
import optuna
import time
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold, train_test_split
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, RobustScaler
import pickle
from collections import defaultdict


### Preparación datasets

In [30]:
# Cargamos los csv de los tifs
path = "saved_files/dataset"
depth = "lt_1"
dfs = {}
all_datasets = False
for archivo in os.listdir(path):
    if all_datasets:
        if f"{depth}" in archivo:
            nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
            ruta_completa = os.path.join(path, archivo)
            dfs[nombre_sin_extension] = pd.read_csv(ruta_completa)
    else:
        if archivo.endswith(f"{depth}_features.csv"):
            nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
            ruta_completa = os.path.join(path, archivo)
            dfs[nombre_sin_extension[:-9]] = pd.read_csv(ruta_completa)

In [22]:
dfs.keys()

dict_keys(['C2RCC_rhown_5x5_depth_gt_3', 'C2X_rhown_5x5_depth_gt_3', 'C2X_rhow_3x3_depth_gt_3', 'C2X_rhow_9x9_depth_gt_3', 'C2X_rhown_1x1_depth_gt_3', 'TOA_9x9_depth_gt_3', 'C2X-Complex_rhow_3x3_depth_gt_3', 'C2X-Complex_rhown_5x5_depth_gt_3', 'C2RCC_rhow_3x3_depth_gt_3', 'C2X-Complex_rhown_1x1_depth_gt_3', 'TOA_5x5_depth_gt_3', 'C2X-Complex_rhow_9x9_depth_gt_3', 'C2RCC_rhow_5x5_depth_gt_3', 'TOA_1x1_depth_gt_3', 'C2X-Complex_rhown_3x3_depth_gt_3', 'C2X_rhown_9x9_depth_gt_3', 'C2X_rhow_5x5_depth_gt_3', 'C2RCC_rhown_3x3_depth_gt_3', 'C2X-Complex_rhow_5x5_depth_gt_3', 'C2RCC_rhow_1x1_depth_gt_3', 'C2RCC_rhow_9x9_depth_gt_3', 'C2RCC_rhown_9x9_depth_gt_3', 'TOA_3x3_depth_gt_3', 'C2X-Complex_rhow_1x1_depth_gt_3', 'C2X_rhown_3x3_depth_gt_3', 'C2X-Complex_rhown_9x9_depth_gt_3', 'C2RCC_rhown_1x1_depth_gt_3', 'C2X_rhow_1x1_depth_gt_3'])

In [194]:
dfs["C2RCC_rhow_1x1_depth_lt_1"]

,Date,Buoy,Latitude,Longitude,rhow_B1,rhow_B2,rhow_B3,rhow_B4,rhow_B5,rhow_B6,...,dif_rel_4bands_rhow_B2_B5_B3_B4,dif_rel_4bands_rhow_B2_B5_B4_B3,dif_rel_4bands_rhow_B3_B2_B4_B5,dif_rel_4bands_rhow_B3_B2_B5_B4,dif_rel_4bands_rhow_B3_B4_B5_B2,dif_rel_4bands_rhow_B3_B5_B4_B2,dif_rel_4bands_rhow_B4_B2_B5_B3,dif_rel_4bands_rhow_B4_B3_B5_B2,sum_norm_3bands_rhow_B2_B4_B3,sum_norm_3bands_rhow_B3_B5_B4
0,2016-08-09,CTD1,4187246,695025,0.010973,0.017248,0.033436,0.016780,0.012437,0.003690,...,-0.606,0.885,0.589,1.197,1.272,1.716,0.601,-0.219,0.671,0.914
1,2016-08-09,CTD2,4181518,693105,0.007180,0.010530,0.019375,0.011021,0.008281,0.002545,...,-0.486,0.703,0.509,1.089,0.972,1.293,0.619,-0.218,0.721,0.910
2,2016-08-09,CTD3,4181698,695238,0.007966,0.011565,0.020111,0.010630,0.007752,0.002311,...,-0.400,0.963,0.368,1.010,1.222,1.675,0.534,-0.142,0.701,0.906
3,2016-08-09,CTD4,4180266,698264,0.008062,0.010310,0.014764,0.008631,0.006252,0.001894,...,-0.062,1.064,0.052,0.708,1.104,1.524,0.414,-0.022,0.755,0.898
4,2016-08-09,CTD6,4176009,695829,0.008603,0.013129,0.023304,0.010864,0.007707,0.002229,...,-0.442,1.237,0.365,1.066,1.558,2.196,0.497,-0.121,0.659,0.908
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
446,2023-05-25,CTD8,4174178,693048,0.016512,0.029853,0.043600,0.007112,0.004319,0.001135,...,0.781,6.749,-0.186,0.853,5.986,9.856,0.139,0.018,0.503,0.945
447,2023-05-25,CTD9,4171106,693183,0.007081,0.012752,0.026582,0.009738,0.006888,0.001995,...,-0.879,1.485,0.671,1.377,2.190,3.095,0.505,-0.174,0.572,0.922
448,2023-05-25,CTD10,4170388,695646,0.007636,0.014290,0.026969,0.006672,0.004333,0.001186,...,-0.745,3.050,0.348,1.238,3.739,5.756,0.306,-0.056,0.508,0.930
449,2023-05-25,CTD11,4169609,700351,0.007547,0.014033,0.026831,0.007324,0.004826,0.001328,...,-0.756,2.635,0.394,1.253,3.320,5.038,0.342,-0.071,0.523,0.927


In [29]:
# Para seleccionar manualmente qué dataframes nos quedamos
# Si leemos por profundidad ya no hace falta esto
dfs_to_keep = [
    "C2X_1x1_merge_depth_lt_1", "C2X_3x3_merge_depth_lt_1", "C2X_5x5_merge_depth_lt_1", 
    "C2X-Complex_1x1_merge_depth_lt_1", "C2X-Complex_3x3_merge_depth_lt_1", "C2X-Complex_5x5_merge_depth_lt_1",
    "C2RCC_1x1_merge_depth_lt_1", "C2RCC_3x3_merge_depth_lt_1", "C2RCC_5x5_merge_depth_lt_1",
    "TOA_1x1_merge_depth_lt_1", "TOA_3x3_merge_depth_lt_1", "TOA_5x5_merge_depth_lt_1",
]

#dfs = {k: dfs[k] for k in dfs_to_keep if k in dfs}

In [31]:

# Limpiamos valores nulos
for nombre_df, df in dfs.items():
    for band_set in ["rhow", "rhown","rtoa"]:
        dfs[nombre_df] = df.dropna()

In [32]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'Invierno'
    elif month in [3, 4, 5]:
        return 'Primavera'
    elif month in [6, 7, 8]:
        return 'Verano'
    else:
        return 'Otoño'
    

def get_zone(buoy):
    if buoy in ["CTD1", "CTD2", "CTD3", "CTD4"]:
        return 'Zona-1'
    elif buoy in ["CTD6", "CTD8", "CTD9", "CTD10", "CTD12"]:
        return 'Zona-2'
    elif buoy in ["CTD7"]:
        return 'Zona-3'
    elif buoy in ["CTD11"]:
        return 'Zona-4'

In [33]:

for nombre_df, df in dfs.items():
    # Marcamos las columnas de CHl alta (equivalente a quantile(0.93))
    df["High_Chl"] = df["Chl"]>5
    # Sacamos la estación de cada fecha
    df['Date'] = pd.to_datetime(df['Date'])
    df['Season'] = df['Date'].dt.month.apply(get_season)
    
    # Etiquetamos la zona de la observación (comentado porque para aplicar el modelo habría que segmentar todo el Mar Menor - se puede hacer por px)
    # df['Zone'] = df['Buoy'].apply(get_zone)
    # Ponemos las columnas como categóricas, para Season y Zone
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype('category')

    df = pd.concat([df.drop(columns=["Season"]),pd.get_dummies(df["Season"])], axis=1)
    dfs[nombre_df] = df
    

In [53]:
# Filtro para quitar columnas muy correlacionadas de cada dataset - de momento no lo usamos
def filtrar_columnas(df):

    # 1. Separar columnas numéricas y no numéricas
    df_numericas = df.select_dtypes(include='number')
    df_no_numericas = df.select_dtypes(exclude='number')

    # 2. Calcular la correlación con 'Chl' solo entre columnas numéricas
    correlaciones = df_numericas.corr()['Chl'].drop('Chl')

    # 3. Filtrar predictores numéricos con correlación significativa
    umbral_corr = 0.1
    columnas_utiles = correlaciones[correlaciones.abs() >= umbral_corr].index.tolist()

    # 4. Reconstruir el DataFrame con:
    # - Las columnas numéricas útiles
    # - La columna objetivo 'Chl'
    df_filtrado = pd.concat([df[columnas_utiles + ['Chl']]], axis=1)

    # Calcular la matriz de correlación entre predictores
    corr_matrix = df_filtrado.drop(columns='Chl').corr().abs()

    # Seleccionar columnas a eliminar (altamente correlacionadas entre sí)
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    columnas_redundantes = [col for col in upper.columns if any(upper[col] > 0.98)]

    # Eliminar redundantes
    df_final = pd.concat([df_filtrado.drop(columns=columnas_redundantes), df_no_numericas], axis=1)
    df_final = df_final.drop(columns=["Date", "Buoy"])

    return df_final


# for nombre_df, df in dfs.items():
#     dfs[nombre_df] = filtrar_columnas(df)

In [34]:
model_params ={
    "XGB" : {
        'n_estimators': 1000,
        'learning_rate': 0.01,
        'max_depth': 7,
        'min_child_weight': 3,
        'subsample': 0.9,
        'colsample_bytree': 0.9,
        'device': 'cpu',
        'objective': 'reg:squarederror',
        'tree_method': 'hist',
        'enable_categorical': True,
        #'early_stopping_rounds': 50,
        'eval_metric': 'rmse'
        },

    "LBM" : {
        'learning_rate': 0.05,
        'num_leaves': 20,
        'max_depth': 7,
        'min_child_samples': 3,
        'subsample': 0.6,
        'colsample_bytree': 0.9,
        'n_estimators': 1000,
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'device': 'cpu',  
        'verbosity': -1,
        #'early_stopping_rounds': 50
        },

    "MLP": {
        'hidden_layer_sizes': (100,),
        'activation': 'relu',
        'solver': 'adam',
        'alpha': 0.0001,
        'learning_rate': 'constant',
        'learning_rate_init': 0.001,
        'max_iter': 200,
        'shuffle': True,
        'random_state': None,
        'tol': 1e-4,
        'n_iter_no_change': 25,
        'verbose': False,
        'early_stopping': True,
        'validation_fraction': 0.2
        },

    "SVR": {
        'kernel': 'sigmoid',        
        'C': 1.0,               
        'epsilon': 0.1,           
        'gamma': 'scale',        
        'shrinking': True,
        'tol': 1e-3,
        'max_iter': -1,          
        'verbose': False,
    },

    "KNN": {
        'n_neighbors': 5,
        'weights': 'uniform',      
        'algorithm': 'auto',      
        'leaf_size': 30,
        'p': 2,                    
        'metric': 'minkowski',
        'n_jobs': -1             
    },

    "RF": {
        'n_estimators': 100,         
        'criterion': 'squared_error',
        'max_depth': 10,         
        'min_samples_split': 2,
        'min_samples_leaf': 1,    
        'bootstrap': True,
        'random_state': 42,
        'verbose': 0
    },

    "CAT": {
        'iterations': 1000,
        'learning_rate': 0.03,
        'depth': 6,
        'l2_leaf_reg': 3.0,
        'loss_function': 'RMSE',
        'eval_metric': 'RMSE',
        'random_seed': 42,
        'allow_writing_files': False,
        'early_stopping_rounds': 50,
        'verbose': False
    },

    "EN": {
        'alpha': 1.0,              # fuerza de regularización
        'l1_ratio': 0.5,           # mezcla entre L1 (lasso) y L2 (ridge)
        'fit_intercept': True,
        'max_iter': 1000,
        'tol': 1e-4,
        'selection': 'cyclic',
        'random_state': 42
    }

}


models = {
    "XGB": XGBRegressor(**model_params['XGB']),
    "LBM": LGBMRegressor(**model_params['LBM']),
    "MLP": MLPRegressor(**model_params['MLP']),
    #"SVR": SVR(**model_params['SVR']),
    "KNN": KNeighborsRegressor(**model_params['KNN']),
    #"LR": LinearRegression().
    "RF": RandomForestRegressor(**model_params['RF']),
    "CAT": CatBoostRegressor(**model_params["CAT"]),
    "EN":  ElasticNet(**model_params["EN"])
}

### Entrenamiento con parámetros por defecto

In [35]:
results = {}

for nombre_df, df in list(dfs.items()):
    #print(nombre_df)
    df = df.iloc[:,4:]

    # Para usar solamente bandas, sin combinaciones
    # if 'TOA' in nombre_df:
    #     # TOA solamente con las bandas, parece que las combinaciones solo meten ruido
    #     df = df.iloc[:,np.r_[0:14, 58:60]]
    # if 'rhow' in nombre_df and 'rhown' not in nombre_df:
    #     df = df.iloc[:,np.r_[0:9, 53:55]]
    # if 'rhown' in nombre_df:
    #     df = df.iloc[:,np.r_[0:7, 51:53]]

    train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df["High_Chl"]) # TEST 20% TRAIN 80%
    target = "Chl"

    train, val = train_test_split(train, test_size=0.25, random_state=42, stratify=train["High_Chl"]) # TRAIN 60% VAL 20% TEST%

    X_train = train.drop(columns=[target,"High_Chl"])
    X_val = val.drop(columns=[target, "High_Chl"])
    X_test = test.drop(columns=[target, "High_Chl"])
    y_train = train[target]
    y_val = val[target]
    y_test = test[target]

    
    scaler_X = RobustScaler()
    scaler_y = RobustScaler()
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_val_scaled = scaler_X.transform(X_val)
    X_test_scaled = scaler_X.transform(X_test)
    y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
    
    results[nombre_df] = {name: {'RMSE': None, 'R2': None} for name in models}
    val_preds = {}
    test_preds = {}

    for name, model in models.items():
        print(f"Fitting {name} for {nombre_df}")
        if name in ["MLP", "SVR", "KNN", "LR", "EN"]:
            model.fit(X_train_scaled, y_train_scaled)
            #test_pred = model.predict(X_test_scaled)
            val_pred = scaler_y.inverse_transform(model.predict(X_val_scaled).reshape(-1, 1)).ravel()
            test_pred = scaler_y.inverse_transform(model.predict(X_test_scaled).reshape(-1, 1)).ravel()
        else:
            model.fit(X_train, y_train)
            val_pred = model.predict(X_val)
            test_pred = model.predict(X_test)

        val_preds[name] = val_pred
        test_preds[name] = test_pred

        rmse = np.sqrt(mean_squared_error(y_test, test_pred))
        r2 = r2_score(y_test, test_pred)

        results[nombre_df][name]['RMSE'] = rmse.round(2)
        results[nombre_df][name]['R2'] = r2.round(2)

    # Meta-modelo
    meta_X = np.vstack([val_preds[model] for model in models]).T
    meta_y = y_val.values
    meta_model = Ridge().fit(meta_X, meta_y)

    # Predicción final ensemble
    test_meta_X = np.vstack([test_preds[model] for model in models]).T
    ensemble_pred = meta_model.predict(test_meta_X)

    rmse_ens = np.sqrt(mean_squared_error(y_test, ensemble_pred))
    r2_ens = r2_score(y_test, ensemble_pred)

    results[nombre_df]["Ensemble"] = {
        "RMSE": round(rmse_ens, 2),
        "R2": round(r2_ens, 2)
    }

Fitting XGB for C2RCC_rhow_5x5_depth_lt_1
Fitting LBM for C2RCC_rhow_5x5_depth_lt_1
Fitting MLP for C2RCC_rhow_5x5_depth_lt_1
Fitting KNN for C2RCC_rhow_5x5_depth_lt_1
Fitting RF for C2RCC_rhow_5x5_depth_lt_1
Fitting CAT for C2RCC_rhow_5x5_depth_lt_1
Fitting EN for C2RCC_rhow_5x5_depth_lt_1
Fitting XGB for C2X-Complex_rhown_3x3_depth_lt_1
Fitting LBM for C2X-Complex_rhown_3x3_depth_lt_1
Fitting MLP for C2X-Complex_rhown_3x3_depth_lt_1
Fitting KNN for C2X-Complex_rhown_3x3_depth_lt_1
Fitting RF for C2X-Complex_rhown_3x3_depth_lt_1
Fitting CAT for C2X-Complex_rhown_3x3_depth_lt_1
Fitting EN for C2X-Complex_rhown_3x3_depth_lt_1
Fitting XGB for TOA_9x9_depth_lt_1
Fitting LBM for TOA_9x9_depth_lt_1
Fitting MLP for TOA_9x9_depth_lt_1


/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Fitting KNN for TOA_9x9_depth_lt_1
Fitting RF for TOA_9x9_depth_lt_1
Fitting CAT for TOA_9x9_depth_lt_1
Fitting EN for TOA_9x9_depth_lt_1
Fitting XGB for C2X_rhow_3x3_depth_lt_1
Fitting LBM for C2X_rhow_3x3_depth_lt_1
Fitting MLP for C2X_rhow_3x3_depth_lt_1
Fitting KNN for C2X_rhow_3x3_depth_lt_1
Fitting RF for C2X_rhow_3x3_depth_lt_1
Fitting CAT for C2X_rhow_3x3_depth_lt_1
Fitting EN for C2X_rhow_3x3_depth_lt_1
Fitting XGB for C2RCC_rhow_9x9_depth_lt_1
Fitting LBM for C2RCC_rhow_9x9_depth_lt_1
Fitting MLP for C2RCC_rhow_9x9_depth_lt_1


/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Fitting KNN for C2RCC_rhow_9x9_depth_lt_1
Fitting RF for C2RCC_rhow_9x9_depth_lt_1
Fitting CAT for C2RCC_rhow_9x9_depth_lt_1
Fitting EN for C2RCC_rhow_9x9_depth_lt_1
Fitting XGB for C2X-Complex_rhown_5x5_depth_lt_1
Fitting LBM for C2X-Complex_rhown_5x5_depth_lt_1
Fitting MLP for C2X-Complex_rhown_5x5_depth_lt_1
Fitting KNN for C2X-Complex_rhown_5x5_depth_lt_1
Fitting RF for C2X-Complex_rhown_5x5_depth_lt_1
Fitting CAT for C2X-Complex_rhown_5x5_depth_lt_1
Fitting EN for C2X-Complex_rhown_5x5_depth_lt_1
Fitting XGB for C2X_rhown_9x9_depth_lt_1
Fitting LBM for C2X_rhown_9x9_depth_lt_1
Fitting MLP for C2X_rhown_9x9_depth_lt_1
Fitting KNN for C2X_rhown_9x9_depth_lt_1
Fitting RF for C2X_rhown_9x9_depth_lt_1
Fitting CAT for C2X_rhown_9x9_depth_lt_1
Fitting EN for C2X_rhown_9x9_depth_lt_1
Fitting XGB for TOA_3x3_depth_lt_1
Fitting LBM for TOA_3x3_depth_lt_1
Fitting MLP for TOA_3x3_depth_lt_1
Fitting KNN for TOA_3x3_depth_lt_1
Fitting RF for TOA_3x3_depth_lt_1
Fitting CAT for TOA_3x3_depth_lt_1

...

Fitting KNN for C2RCC_rhown_5x5_depth_lt_1
Fitting RF for C2RCC_rhown_5x5_depth_lt_1
Fitting CAT for C2RCC_rhown_5x5_depth_lt_1
Fitting EN for C2RCC_rhown_5x5_depth_lt_1
Fitting XGB for TOA_1x1_depth_lt_1
Fitting LBM for TOA_1x1_depth_lt_1
Fitting MLP for TOA_1x1_depth_lt_1
Fitting KNN for TOA_1x1_depth_lt_1
Fitting RF for TOA_1x1_depth_lt_1
Fitting CAT for TOA_1x1_depth_lt_1
Fitting EN for TOA_1x1_depth_lt_1
Fitting XGB for C2X-Complex_rhow_5x5_depth_lt_1
Fitting LBM for C2X-Complex_rhow_5x5_depth_lt_1
Fitting MLP for C2X-Complex_rhow_5x5_depth_lt_1
Fitting KNN for C2X-Complex_rhow_5x5_depth_lt_1
Fitting RF for C2X-Complex_rhow_5x5_depth_lt_1
Fitting CAT for C2X-Complex_rhow_5x5_depth_lt_1
Fitting EN for C2X-Complex_rhow_5x5_depth_lt_1
Fitting XGB for C2X-Complex_rhow_9x9_depth_lt_1
Fitting LBM for C2X-Complex_rhow_9x9_depth_lt_1
Fitting MLP for C2X-Complex_rhow_9x9_depth_lt_1
Fitting KNN for C2X-Complex_rhow_9x9_depth_lt_1
Fitting RF for C2X-Complex_rhow_9x9_depth_lt_1
Fitting CAT fo

/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Fitting KNN for C2RCC_rhown_3x3_depth_lt_1
Fitting RF for C2RCC_rhown_3x3_depth_lt_1
Fitting CAT for C2RCC_rhown_3x3_depth_lt_1
Fitting EN for C2RCC_rhown_3x3_depth_lt_1
Fitting XGB for C2RCC_rhow_1x1_depth_lt_1
Fitting LBM for C2RCC_rhow_1x1_depth_lt_1
Fitting MLP for C2RCC_rhow_1x1_depth_lt_1


/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Fitting KNN for C2RCC_rhow_1x1_depth_lt_1
Fitting RF for C2RCC_rhow_1x1_depth_lt_1
Fitting CAT for C2RCC_rhow_1x1_depth_lt_1
Fitting EN for C2RCC_rhow_1x1_depth_lt_1


In [36]:
# Para hacer un dataframe con multiindex en el que se vea todo

rows = []

for df_name, model_scores in results.items():
    row = {}
    for model_name, metrics in model_scores.items():
        for metric_name, value in metrics.items():
            # Clave: (métrica, modelo) → orden correcto para columnas
            row[(metric_name, model_name)] = value
    rows.append((df_name, row))

df_results = pd.DataFrame.from_dict(dict(rows), orient="index")
df_results.columns = pd.MultiIndex.from_tuples(df_results.columns, names=["Metric", "Model"])
df_results = df_results.sort_index(axis=1, level=0)
df_results = df_results.sort_index(axis=0)

In [37]:
df_results

Metric                              R2                                         \
Model                              CAT    EN Ensemble   KNN   LBM   MLP    RF   
C2RCC_rhow_1x1_depth_lt_1         0.55  0.32     0.76  0.65  0.52  0.71  0.67   
C2RCC_rhow_3x3_depth_lt_1         0.65  0.33     0.60  0.63  0.42  0.48  0.62   
C2RCC_rhow_5x5_depth_lt_1         0.67  0.37     0.75  0.64  0.64  0.65  0.70   
C2RCC_rhow_9x9_depth_lt_1         0.79  0.37     0.78  0.62  0.76  0.70  0.74   
C2RCC_rhown_1x1_depth_lt_1        0.52  0.30     0.58  0.68  0.65  0.28  0.65   
C2RCC_rhown_3x3_depth_lt_1        0.59  0.31     0.69  0.61  0.59  0.64  0.58   
C2RCC_rhown_5x5_depth_lt_1        0.66  0.36     0.71  0.67  0.59  0.74  0.67   
C2RCC_rhown_9x9_depth_lt_1        0.82  0.36     0.77  0.62  0.71  0.71  0.74   
C2X-Complex_rhow_1x1_depth_lt_1   0.68  0.36     0.67  0.64  0.65  0.55  0.60   
C2X-Complex_rhow_3x3_depth_lt_1   0.72  0.37     0.66  0.60  0.60  0.62  0.61   
C2X-Complex_rhow_5x5_depth_lt_1   0.71  0.41     0.59  0.58  0.56  0.57  0.57   
C2X-Complex_rhow_9x9_depth_lt_1   0.80  0.50     0.69  0.64  0.73  0.61  0.69   
C2X-Complex_rhown_1x1_depth_lt_1  0.69  0.40     0.66  0.69  0.61  0.63  0.59   
C2X-Complex_rhown_3x3_depth_lt_1  0.68  0.37     0.62  0.56  0.54  0.39  0.57   
C2X-Complex_rhown_5x5_depth_lt_1  0.73  0.44     0.64  0.62  0.65  0.23  0.65   
C2X-Complex_rhown_9x9_depth_lt_1  0.77  0.50     0.71  0.67  0.61  0.64  0.66   
C2X_rhow_1x1_depth_lt_1           0.69  0.62     0.80  0.60  0.78  0.59  0.62   
C2X_rhow_3x3_depth_lt_1           0.64  0.47     0.45  0.62  0.49  0.43  0.57   
C2X_rhow_5x5_depth_lt_1           0.52  0.54     0.62  0.57  0.44  0.46  0.45   
C2X_rhow_9x9_depth_lt_1           0.70  0.65     0.67  0.70  0.70  0.65  0.68   
C2X_rhown_1x1_depth_lt_1          0.58  0.53     0.52  0.50  0.40  0.59  0.51   
C2X_rhown_3x3_depth_lt_1          0.60  0.32     0.67  0.51  0.38  0.59  0.53   
C2X_rhown_5x5_depth_lt_1          0.52  0.43     0.50  0.45  0.13  0.37  0.43   
C2X_rhown_9x9_depth_lt_1          0.65  0.60     0.61  0.55  0.56  0.66  0.63   
TOA_1x1_depth_lt_1                0.35  0.01     0.37  0.37  0.18  0.11  0.14   
TOA_3x3_depth_lt_1                0.60  0.02     0.44  0.42  0.43  0.53  0.48   
TOA_5x5_depth_lt_1                0.59  0.04     0.45  0.46  0.41  0.22  0.54   
TOA_9x9_depth_lt_1                0.60  0.04     0.49  0.46  0.38  0.50  0.42   

Metric                                  RMSE                                   \
Model                              XGB   CAT    EN Ensemble   KNN   LBM   MLP   
C2RCC_rhow_1x1_depth_lt_1         0.51  3.18  3.93     2.36  2.82  3.30  2.55   
C2RCC_rhow_3x3_depth_lt_1         0.57  2.80  3.92     3.00  2.90  3.64  3.43   
C2RCC_rhow_5x5_depth_lt_1         0.72  2.73  3.79     2.41  2.86  2.86  2.84   
C2RCC_rhow_9x9_depth_lt_1         0.83  2.17  3.80     2.24  2.96  2.33  2.61   
C2RCC_rhown_1x1_depth_lt_1        0.53  3.31  3.99     3.10  2.70  2.83  4.04   
C2RCC_rhown_3x3_depth_lt_1        0.75  3.04  3.96     2.67  2.96  3.06  2.86   
C2RCC_rhown_5x5_depth_lt_1        0.66  2.76  3.82     2.55  2.74  3.06  2.42   
C2RCC_rhown_9x9_depth_lt_1        0.84  2.05  3.82     2.29  2.93  2.56  2.58   
C2X-Complex_rhow_1x1_depth_lt_1   0.59  2.72  3.81     2.75  2.87  2.82  3.20   
C2X-Complex_rhow_3x3_depth_lt_1   0.63  2.53  3.79     2.79  3.03  3.00  2.96   
C2X-Complex_rhow_5x5_depth_lt_1   0.58  2.56  3.65     3.07  3.09  3.16  3.13   
C2X-Complex_rhow_9x9_depth_lt_1   0.63  2.13  3.38     2.64  2.84  2.47  2.96   
C2X-Complex_rhown_1x1_depth_lt_1  0.58  2.64  3.69     2.77  2.65  3.00  2.92   
C2X-Complex_rhown_3x3_depth_lt_1  0.65  2.71  3.77     2.92  3.18  3.22  3.73   
C2X-Complex_rhown_5x5_depth_lt_1  0.69  2.46  3.58     2.87  2.94  2.83  4.20   
C2X-Complex_rhown_9x9_depth_lt_1  0.62  2.30  3.38     2.55  2.74  2.96  2.85   
C2X_rhow_1x1_depth_lt_1           0.66  2.64  2.94     2.11  3.01  2.21  3.06   
C2X_rhow_3x3_depth_lt_1       

In [243]:
df_results

Metric                              R2                                         \
Model                              CAT    EN Ensemble   KNN   LBM    LR   MLP   
C2RCC_rhow_1x1_depth_lt_1         0.60  0.32     0.77  0.65  0.62  0.64  0.78   
C2RCC_rhow_3x3_depth_lt_1         0.62  0.33     0.79  0.63  0.63  0.47  0.78   
C2RCC_rhow_5x5_depth_lt_1         0.67  0.37     0.78  0.64  0.65  0.64  0.80   
C2RCC_rhow_9x9_depth_lt_1         0.78  0.37     0.69  0.62  0.70  0.49  0.77   
C2RCC_rhown_1x1_depth_lt_1        0.51  0.30     0.72  0.68  0.62  0.44  0.77   
C2RCC_rhown_3x3_depth_lt_1        0.62  0.31     0.78  0.61  0.62  0.48  0.76   
C2RCC_rhown_5x5_depth_lt_1        0.64  0.36     0.69  0.67  0.64  0.62  0.81   
C2RCC_rhown_9x9_depth_lt_1        0.81  0.36     0.74  0.62  0.70  0.55  0.77   
C2X-Complex_rhow_1x1_depth_lt_1   0.68  0.36     0.43  0.64  0.70  0.15  0.72   
C2X-Complex_rhow_3x3_depth_lt_1   0.71  0.37     0.51  0.60  0.56  0.10  0.72   
C2X-Complex_rhow_5x5_depth_lt_1   0.71  0.41     0.63  0.58  0.67 -2.23  0.73   
C2X-Complex_rhow_9x9_depth_lt_1   0.80  0.50     0.74  0.64  0.81  0.41  0.77   
C2X-Complex_rhown_1x1_depth_lt_1  0.73  0.40     0.77  0.69  0.68  0.55  0.72   
C2X-Complex_rhown_3x3_depth_lt_1  0.67  0.37     0.53  0.56  0.60  0.54  0.72   
C2X-Complex_rhown_5x5_depth_lt_1  0.71  0.44     0.78  0.62  0.65  0.49  0.72   
C2X-Complex_rhown_9x9_depth_lt_1  0.79  0.50     0.79  0.67  0.74  0.45  0.80   
C2X_rhow_1x1_depth_lt_1           0.68  0.62     0.74  0.60  0.59  0.08  0.55   
C2X_rhow_3x3_depth_lt_1           0.62  0.47     0.52  0.62  0.59 -0.88  0.65   
C2X_rhow_5x5_depth_lt_1           0.48  0.54     0.69  0.57  0.50  0.54  0.76   
C2X_rhow_9x9_depth_lt_1           0.68  0.65     0.70  0.70  0.67  0.37  0.79   
C2X_rhown_1x1_depth_lt_1          0.61  0.53     0.59  0.50  0.52  0.33  0.53   
C2X_rhown_3x3_depth_lt_1          0.61  0.32     0.69  0.51  0.60  0.03  0.66   
C2X_rhown_5x5_depth_lt_1          0.50  0.43     0.60  0.45  0.43  0.72  0.71   
C2X_rhown_9x9_depth_lt_1          0.68  0.60     0.76  0.55  0.59  0.47  0.77   
TOA_1x1_depth_lt_1                0.36  0.01     0.24  0.37  0.45  0.27  0.41   
TOA_3x3_depth_lt_1                0.63  0.02     0.45  0.42  0.51  0.45  0.50   
TOA_5x5_depth_lt_1                0.60  0.04     0.49  0.46  0.42  0.38  0.57   
TOA_9x9_depth_lt_1                0.60  0.04     0.42  0.46  0.54 -0.03  0.68   

Metric                                              RMSE                       \
Model                               RF   SVR   XGB   CAT    EN Ensemble   KNN   
C2RCC_rhow_1x1_depth_lt_1         0.62  0.30  0.39  3.00  3.93     2.28  2.82   
C2RCC_rhow_3x3_depth_lt_1         0.61  0.33  0.58  2.94  3.92     2.20  2.90   
C2RCC_rhow_5x5_depth_lt_1         0.70  0.35  0.78  2.72  3.79     2.23  2.86   
C2RCC_rhow_9x9_depth_lt_1         0.71  0.33  0.65  2.26  3.80     2.66  2.96   
C2RCC_rhown_1x1_depth_lt_1        0.56  0.30  0.58  3.33  3.99     2.54  2.70   
C2RCC_rhown_3x3_depth_lt_1        0.59  0.32  0.76  2.94  3.96     2.25  2.96   
C2RCC_rhown_5x5_depth_lt_1        0.71  0.35  0.77  2.85  3.82     2.65  2.74   
C2RCC_rhown_9x9_depth_lt_1        0.76  0.33  0.82  2.07  3.82     2.43  2.93   
C2X-Complex_rhow_1x1_depth_lt_1   0.60  0.23  0.66  2.71  3.81     3.59  2.87   
C2X-Complex_rhow_3x3_depth_lt_1   0.62  0.28  0.59  2.57  3.79     3.33  3.03   
C2X-Complex_rhow_5x5_depth_lt_1   0.58  0.29  0.59  2.56  3.65     2.90  3.09   
C2X-Complex_rhow_9x9_depth_lt_1   0.68  0.22  0.53  2.14  3.38     2.43  2.84   
C2X-Complex_rhown_1x1_depth_lt_1  0.59  0.33  0.61  2.49  3.69     2.27  2.65   
C2X-Complex_rhown_3x3_depth_lt_1  0.53  0.36  0.64  2.75  3.77     3.27  3.18   
C2X-Complex_rhown_5x5_depth_lt_1  0.65  0.32  0.73  2.55  3.58     2.23  2.94   
C2X-Complex_rhown_9x9_depth_lt_1  0.69  0.29  0.53  2.17  3.38     2.17  2.74   
C2X_rhow_1x1_depth_lt_1           0.62  0.23  0.60  2.68  2.94     2.45  3.01   
C2X_rhow_3x3_depth_lt_1       

### Optimización de hiperparámetros

In [292]:
def objective(trial, df, target, model_name):
    df = df.iloc[:,4:]
    # Mismos splits que para el entrenamiento
    train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df["High_Chl"]) # TEST 20% TRAIN 80%
    target = "Chl"
    train, val = train_test_split(train, test_size=0.25, random_state=42, stratify=train["High_Chl"])

    X = train.drop(columns=[target, 'High_Chl'])
    y = train[target]
    y_class = train["High_Chl"]

    # Definimos los folds
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    oof_preds = np.zeros(len(train))  # Almacenar las predicciones OOF

    start = time.time()
    
    if model_name == "LBM":
        params = {
            'objective': 'regression',
            'metric': 'rmse',
            'boosting_type': 'gbdt',
            'device': 'cpu',  # usa CPU/GPU
            'verbosity': -1,
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'num_leaves': trial.suggest_categorical('num_leaves', [20, 40, 60, 80]),
            'max_depth': trial.suggest_int('max_depth', 5, 8),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 25),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'n_estimators': trial.suggest_categorical('n_estimators', [500, 1000, 2000])
        }

    if model_name == "XGB":
        params = {
            'n_estimators': trial.suggest_categorical('n_estimators', [500, 1000, 2000]),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 5, 8),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 4),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'device': 'cpu',
            'objective': 'reg:squarederror',
            'tree_method': 'hist',
            'enable_categorical': True,
            'eval_metric': 'rmse'
        }
    
    if model_name == "MLP":
        # Así para evitar warning de definir como tuple
        hidden_options = {
            '50': (50,),
            '100': (100,),
            '100_50': (100, 50),
            '128_64': (128, 64)
        }
        key = trial.suggest_categorical('hidden_layer_sizes', list(hidden_options.keys()))
        params = {
            'hidden_layer_sizes': hidden_options[key],
            #'hidden_layer_sizes': trial.suggest_categorical('hidden_layer_sizes', [(50,), (100,), (100, 50), (128, 64)]),
            'activation': trial.suggest_categorical('activation', ['relu', 'tanh']),
            'solver': trial.suggest_categorical('solver', ['adam', 'sgd']),
            'alpha': trial.suggest_float('alpha', 1e-5, 1e-1, log=True),
            'learning_rate': trial.suggest_categorical('learning_rate', ['constant', 'adaptive']),
            'learning_rate_init': trial.suggest_float('learning_rate_init', 1e-4, 1e-2, log=True),
            'max_iter': 200,
            'n_iter_no_change': 25,
            'early_stopping': True,
            'validation_fraction': 0.2,
            'random_state': 42,
            'verbose': False
        }

    if model_name == "SVR":
        params = {
            'kernel': trial.suggest_categorical('kernel', ['rbf', 'sigmoid']),
            'C': trial.suggest_float('C', 0.1, 10.0, log=True),
            'epsilon': trial.suggest_float('epsilon', 0.01, 0.2),
            'gamma': trial.suggest_categorical('gamma', ['scale', 'auto']),
            'shrinking': True,
            'tol': 1e-3,
            'max_iter': -1,
            'verbose': False
        }

    if model_name == "KNN":
        params = {
            'n_neighbors': trial.suggest_int('n_neighbors', 3, 15),
            'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
            'algorithm': 'auto',
            'leaf_size': trial.suggest_int('leaf_size', 10, 40),
            'p': 2,  # 1 = manhattan, 2 = euclídea
            'metric': 'minkowski',
            'n_jobs': -1
        }

    if model_name == "LR":
        # No sirve de mucho, pero por completitud
        params = {
            'fit_intercept': trial.suggest_categorical('fit_intercept', [True, False]),
            'positive': trial.suggest_categorical('positive', [True, False]),
        }

    if model_name == "RF":
        params = {
            'n_estimators': trial.suggest_categorical('n_estimators', [100, 300, 500]),
            'max_depth': trial.suggest_int('max_depth', 5, 15),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
            'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
            'random_state': 42,
            'verbose': 0
        }

    if model_name == "CAT":
        params = {
            'iterations': trial.suggest_categorical('iterations', [500, 1000, 2000]),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'depth': trial.suggest_int('depth', 4, 10),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
            'loss_function': 'RMSE',
            'eval_metric': 'RMSE',
            'random_seed': 42,
            'early_stopping_rounds': 50,
            'verbose': False,
        }

    if model_name == "EN":
        params = {
            'alpha': trial.suggest_float('alpha', 1e-4, 10.0, log=True),
            'l1_ratio': trial.suggest_float('l1_ratio', 0.0, 1.0),
            'fit_intercept': True,
            'max_iter': 1000,
            'tol': 1e-4,
            'selection': 'cyclic',
            'random_state': 42
        }

    # Cargamos el modelo correspondiente
    model = models[model_name](**params)

    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_class)):
        print(f"Fold {fold+1}")
        
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        if model_name in ["MLP", "SVR", "KNN", "LR", "EN"]:
            scaler_X = RobustScaler()
            scaler_y = RobustScaler()
            X_train_scaled = scaler_X.fit_transform(X_train)
            X_val_scaled = scaler_X.transform(X_val)
            y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
            model.fit(X_train_scaled, y_train_scaled)
            y_pred = scaler_y.inverse_transform(model.predict(X_val_scaled).reshape(-1, 1)).ravel()
        else:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
        
        oof_preds[val_idx] = y_pred


    print(f"Running time: {time.time() - start:.1f} sec")
    # Calculamos el RMSE OOF
    rmse_score = np.sqrt(mean_squared_error(y, oof_preds))
    r2 = r2_score(y, oof_preds)
    print(f"OOF RMSE: {rmse_score:.2f} | R2: {r2:.2f}")
    
    return r2

In [293]:
models = {
    "XGB": XGBRegressor,
    "LBM": LGBMRegressor,
    "MLP": MLPRegressor,
    "SVR": SVR,
    "KNN": KNeighborsRegressor,
    "LR": LinearRegression,
    "RF": RandomForestRegressor,
    "CAT": CatBoostRegressor,
    "EN":  ElasticNet
}

def run_optuna(df, target, n_trials, model_name):
    results = {}

    print(f"Buscando mejores hiperparámetros para {model_name}...")
    study = optuna.create_study(direction='maximize')
    study.optimize(lambda trial: objective(trial, df, target, model_name), n_trials=n_trials, timeout= 600)
    
    print(f"\n✅ {model_name} - Mejor R2: {study.best_value:.2f}")
    print(f"📋 Parámetros: {study.best_params}\n")
    
    results[model_name] = {
        'best_params': study.best_params,
        'best_score': study.best_value,
        'study': study
    }
    return results

#df = dfs["C2X_rhown_3x3_depth_lt_1"]

# all_results = {}
n_trials = 25
# for model_name in models.keys():
#     all_results[model_name] = run_optuna(df, "Chl", n_trials, model_name)
global_results = {}

for df_name, df in dfs.items():
    print(f"🔍 Optimizando en {df_name}...")
    for model_name in models.keys():
        key = (df_name, model_name)
        result = run_optuna(df, "Chl", n_trials, model_name)
        global_results[key] = result[model_name]

[I 2025-07-11 16:08:52,346] A new study created in memory with name: no-name-b643c892-28af-4ffa-a3d5-07df1c04bcc9


🔍 Optimizando en C2RCC_rhow_5x5_depth_lt_1...
Buscando mejores hiperparámetros para XGB...
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2025-07-11 16:09:01,324] Trial 0 finished with value: 0.6786606658483075 and parameters: {'n_estimators': 1000, 'learning_rate': 0.03363059712784438, 'max_depth': 5, 'min_child_weight': 2, 'subsample': 0.7354620669213784, 'colsample_bytree': 0.926760985439751}. Best is trial 0 with value: 0.6786606658483075.


Running time: 9.0 sec
OOF RMSE: 1.97 | R2: 0.68
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2025-07-11 16:09:11,214] Trial 1 finished with value: 0.610822465595559 and parameters: {'n_estimators': 2000, 'learning_rate': 0.0849999376336989, 'max_depth': 5, 'min_child_weight': 1, 'subsample': 0.9414025251441294, 'colsample_bytree': 0.9559502088665054}. Best is trial 0 with value: 0.6786606658483075.


...

Fold 2
Fold 3
Fold 4
Fold 5
Running time: 0.2 sec
OOF RMSE: 2.46 | R2: 0.50
Fold 1
Fold 2
Fold 3


[I 2025-07-12 00:15:33,787] Trial 22 finished with value: 0.4987277084446985 and parameters: {'alpha': 0.05609674554093761, 'l1_ratio': 0.8826939390659186}. Best is trial 22 with value: 0.4987277084446985.


Fold 4
Fold 5
Running time: 0.2 sec
OOF RMSE: 2.45 | R2: 0.50
Fold 1
Fold 2
Fold 3
Fold 4


/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.842e+01, tolerance: 2.084e-01
  model = cd_fast.enet_coordinate_descent(
/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.758e+01, tolerance: 2.025e-01
  model = cd_fast.enet_coordinate_descent(
/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations

Fold 5
Running time: 0.2 sec
OOF RMSE: 2.79 | R2: 0.35
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Running time: 0.1 sec
OOF RMSE: 2.54 | R2: 0.46

✅ EN - Mejor R2: 0.50
📋 Parámetros: {'alpha': 0.05609674554093761, 'l1_ratio': 0.8826939390659186}



In [294]:
with open("global_results.pkl", "wb") as f:
    pickle.dump(global_results, f)

In [295]:
global_results

{('C2RCC_rhow_5x5_depth_lt_1',
  'XGB'): {'best_params': {'n_estimators': 1000,
   'learning_rate': 0.008490905014217147,
   'max_depth': 6,
   'min_child_weight': 3,
   'subsample': 0.9117231361249971,
   'colsample_bytree': 0.7381845712236873}, 'best_score': 0.6977084098853871, 'study': <optuna.study.study.Study at 0x769648cc2b80>},
 ('C2RCC_rhow_5x5_depth_lt_1',
  'LBM'): {'best_params': {'learning_rate': 0.02073824304970037,
   'num_leaves': 60,
   'max_depth': 8,
   'min_child_samples': 5,
   'subsample': 0.9938714541767556,
   'colsample_bytree': 0.7278831706716934,
   'n_estimators': 2000}, 'best_score': 0.667321133437436, 'study': <optuna.study.study.Study at 0x76963b8dd490>},
 ('C2RCC_rhow_5x5_depth_lt_1',
  'MLP'): {'best_params': {'hidden_layer_sizes': '128_64',
   'activation': 'tanh',
   'solver': 'adam',
   'alpha': 0.005677603912486301,
   'learning_rate': 'adaptive',
   'learning_rate_init': 0.004569000726524064}, 'best_score': 0.6988355877442272, 'study': <optuna.study

In [297]:
# Diccionario para agrupar parámetros por modelo
params_by_model = defaultdict(list)

# Agrupar best_params por modelo
for (df_name, model_name), result in global_results.items():
    best_params = result["best_params"]
    params_by_model[model_name].append(best_params)

# Crear DataFrames con medias y std por modelo
summary_stats = {}

for model_name, param_list in params_by_model.items():
    df_params = pd.DataFrame(param_list)

    # Filtramos solo columnas numéricas para calcular medias y std
    df_numeric = df_params.select_dtypes(include=[np.number])

    stats = pd.concat([df_numeric.mean().rename("mean"), df_numeric.std().rename("std")], axis=1)
    summary_stats[model_name] = stats

# Mostrar un ejemplo
summary_stats["XGB"]



,mean,std
n_estimators,982.142857,630.727723
learning_rate,0.024349,0.022070
max_depth,6.607143,1.100144
min_child_weight,2.357143,1.282771
subsample,0.739660,0.107569
colsample_bytree,0.737681,0.104036


In [303]:
for model, stats_df in summary_stats.items():
    print(f"\n📊 {model}")
    print(stats_df.round(2))



📊 XGB
                    mean     std
n_estimators      982.14  630.73
learning_rate       0.02    0.02
max_depth           6.61    1.10
min_child_weight    2.36    1.28
subsample           0.74    0.11
colsample_bytree    0.74    0.10

📊 LBM
                      mean     std
learning_rate         0.02    0.02
num_leaves           47.14   22.58
max_depth             6.25    1.17
min_child_samples    12.32    7.07
subsample             0.83    0.12
colsample_bytree      0.74    0.11
n_estimators       1285.71  658.68

📊 MLP
                    mean   std
alpha               0.01  0.03
learning_rate_init  0.00  0.00

📊 SVR
         mean   std
C        9.54  0.88
epsilon  0.11  0.05

📊 KNN
              mean   std
n_neighbors   4.71  2.37
leaf_size    24.36  9.45

📊 LR
Empty DataFrame
Columns: [mean, std]
Index: []

📊 RF
                     mean     std
n_estimators       385.71  148.36
max_depth           10.43    3.13
min_samples_split    4.46    2.06
min_samples_leaf     1.54    0.

In [308]:
for model, param_list in params_by_model.items():
    print(f"\n🔤 Parámetros categóricos de {model}:")
    df = pd.DataFrame(param_list)
    df = df.drop(columns=["dataset", "model"])
    for col in df.select_dtypes(include='object').columns:
        print(f"{col}:\n{df[col].value_counts()}")



🔤 Parámetros categóricos de XGB:

🔤 Parámetros categóricos de LBM:

🔤 Parámetros categóricos de MLP:
hidden_layer_sizes:
hidden_layer_sizes
100       13
128_64     9
100_50     5
50         1
Name: count, dtype: int64
activation:
activation
tanh    16
relu    12
Name: count, dtype: int64
solver:
solver
adam    21
sgd      7
Name: count, dtype: int64
learning_rate:
learning_rate
adaptive    16
constant    12
Name: count, dtype: int64

🔤 Parámetros categóricos de SVR:
kernel:
kernel
rbf    28
Name: count, dtype: int64
gamma:
gamma
scale    21
auto      7
Name: count, dtype: int64

🔤 Parámetros categóricos de KNN:
weights:
weights
distance    25
uniform      3
Name: count, dtype: int64

🔤 Parámetros categóricos de LR:

🔤 Parámetros categóricos de RF:

🔤 Parámetros categóricos de CAT:

🔤 Parámetros categóricos de EN:


**Parámetros seleccionados** (Nos quedamos con los que dice Optuna en EN, KNN, SVR y XGB, y mantenemos los que pusimos por defecto en CAT, MLP, LBM y RF. Esto lo decidimos después de ver los resultados al entrenar todo con los que nos daba Optuna y comparar resultados modelo a modelo.)

In [9]:
model_params ={
    "XGB" : {
        'n_estimators': 1000,
        'learning_rate': 0.02,
        'max_depth': 7,
        'min_child_weight': 2,
        'subsample': 0.7,
        'colsample_bytree': 0.7,
        'device': 'cpu',
        'objective': 'reg:squarederror',
        'tree_method': 'hist',
        'enable_categorical': True,
        #'early_stopping_rounds': 50,
        'eval_metric': 'rmse'
        },

    "LBM" : {
        'learning_rate': 0.04,
        'num_leaves': 20,
        'max_depth': 7,
        'min_child_samples': 4,
        'subsample': 0.7,
        'colsample_bytree': 0.7,
        'n_estimators': 1000,
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'device': 'cpu',  
        'verbosity': -1,
        #'early_stopping_rounds': 50
        },

    "MLP": {
        'hidden_layer_sizes': (100,),
        'activation': 'relu',
        'solver': 'adam',
        'alpha': 0.0001,
        'learning_rate': 'constant',
        'learning_rate_init': 0.001,
        'max_iter': 200,
        'shuffle': True,
        'random_state': None,
        'tol': 1e-4,
        'n_iter_no_change': 25,
        'verbose': False,
        'early_stopping': True,
        'validation_fraction': 0.2
        },

    "SVR": {
        'kernel': 'rbf',        
        'C': 9.5,               
        'epsilon': 0.1,           
        'gamma': 'scale',        
        'shrinking': True,
        'tol': 1e-3,
        'max_iter': -1,          
        'verbose': False,
    },

    "KNN": {
        'n_neighbors': 5,
        'weights': 'distance',      
        'algorithm': 'auto',      
        'leaf_size': 25,
        'p': 2,                    
        'metric': 'minkowski',
        'n_jobs': -1             
    },

    "RF": {
        'n_estimators': 100,         
        'criterion': 'squared_error',
        'max_depth': 10,         
        'min_samples_split': 2,
        'min_samples_leaf': 2,    
        'bootstrap': True,
        'random_state': 42,
        'verbose': 0
    },

    "CAT": {
        'iterations': 1000,
        'learning_rate': 0.03,
        'depth': 6,
        'l2_leaf_reg': 3.0,
        'loss_function': 'RMSE',
        'eval_metric': 'RMSE',
        'random_seed': 42,
        'allow_writing_files': False,
        'early_stopping_rounds': 50,
        'verbose': False
    },

    "EN": {
        'alpha': 0.2,             
        'l1_ratio': 0.5,           
        'fit_intercept': True,
        'max_iter': 1000,
        'tol': 1e-4,
        'selection': 'cyclic',
        'random_state': 42
    }

}


models = {
    "XGB": XGBRegressor(**model_params['XGB']),
    "LBM": LGBMRegressor(**model_params['LBM']),
    "MLP": MLPRegressor(**model_params['MLP']),
    "SVR": SVR(**model_params['SVR']),
    "KNN": KNeighborsRegressor(**model_params['KNN']),
    "LR": LinearRegression(),
    "RF": RandomForestRegressor(**model_params['RF']),
    "CAT": CatBoostRegressor(**model_params["CAT"]),
    "EN":  ElasticNet(**model_params["EN"])
}

**Entrenamiento con los parámetros seleccionados**

In [318]:
results = {}

for nombre_df, df in list(dfs.items()):
    #print(nombre_df)
    df = df.iloc[:,4:]

    # Para usar solamente bandas, sin combinaciones
    # if 'TOA' in nombre_df:
    #     # TOA solamente con las bandas, parece que las combinaciones solo meten ruido
    #     df = df.iloc[:,np.r_[0:14, 58:60]]
    # if 'rhow' in nombre_df and 'rhown' not in nombre_df:
    #     df = df.iloc[:,np.r_[0:9, 53:55]]
    # if 'rhown' in nombre_df:
    #     df = df.iloc[:,np.r_[0:7, 51:53]]

    train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df["High_Chl"]) # TEST 20% TRAIN 80%
    target = "Chl"

    train, val = train_test_split(train, test_size=0.25, random_state=42, stratify=train["High_Chl"]) # TRAIN 60% VAL 20% TEST%

    X_train = train.drop(columns=[target,"High_Chl"])
    X_val = val.drop(columns=[target, "High_Chl"])
    X_test = test.drop(columns=[target, "High_Chl"])
    y_train = train[target]
    y_val = val[target]
    y_test = test[target]

    
    scaler_X = RobustScaler()
    scaler_y = RobustScaler()
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_val_scaled = scaler_X.transform(X_val)
    X_test_scaled = scaler_X.transform(X_test)
    y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
    
    results[nombre_df] = {name: {'RMSE': None, 'R2': None} for name in models}
    val_preds = {}
    test_preds = {}

    for name, model in models.items():
        print(f"Fitting {name} for {nombre_df}")
        if name in ["MLP", "SVR", "KNN", "LR", "EN"]:
            model.fit(X_train_scaled, y_train_scaled)
            #test_pred = model.predict(X_test_scaled)
            val_pred = scaler_y.inverse_transform(model.predict(X_val_scaled).reshape(-1, 1)).ravel()
            test_pred = scaler_y.inverse_transform(model.predict(X_test_scaled).reshape(-1, 1)).ravel()
        else:
            model.fit(X_train, y_train)
            val_pred = model.predict(X_val)
            test_pred = model.predict(X_test)

        val_preds[name] = val_pred
        test_preds[name] = test_pred

        rmse = np.sqrt(mean_squared_error(y_test, test_pred))
        r2 = r2_score(y_test, test_pred)

        results[nombre_df][name]['RMSE'] = rmse.round(2)
        results[nombre_df][name]['R2'] = r2.round(2)

    # Meta-modelo
    meta_X = np.vstack([val_preds[model] for model in models]).T
    meta_y = y_val.values
    meta_model = Ridge().fit(meta_X, meta_y)

    # Predicción final ensemble
    test_meta_X = np.vstack([test_preds[model] for model in models]).T
    ensemble_pred = meta_model.predict(test_meta_X)

    rmse_ens = np.sqrt(mean_squared_error(y_test, ensemble_pred))
    r2_ens = r2_score(y_test, ensemble_pred)

    results[nombre_df]["Ensemble"] = {
        "RMSE": round(rmse_ens, 2),
        "R2": round(r2_ens, 2)
    }

Fitting XGB for C2RCC_rhow_5x5_depth_lt_1
Fitting LBM for C2RCC_rhow_5x5_depth_lt_1
Fitting MLP for C2RCC_rhow_5x5_depth_lt_1
Fitting SVR for C2RCC_rhow_5x5_depth_lt_1
Fitting KNN for C2RCC_rhow_5x5_depth_lt_1
Fitting LR for C2RCC_rhow_5x5_depth_lt_1
Fitting RF for C2RCC_rhow_5x5_depth_lt_1
Fitting CAT for C2RCC_rhow_5x5_depth_lt_1
Fitting EN for C2RCC_rhow_5x5_depth_lt_1
Fitting XGB for C2X-Complex_rhown_3x3_depth_lt_1
Fitting LBM for C2X-Complex_rhown_3x3_depth_lt_1
Fitting MLP for C2X-Complex_rhown_3x3_depth_lt_1
Fitting SVR for C2X-Complex_rhown_3x3_depth_lt_1
Fitting KNN for C2X-Complex_rhown_3x3_depth_lt_1
Fitting LR for C2X-Complex_rhown_3x3_depth_lt_1
Fitting RF for C2X-Complex_rhown_3x3_depth_lt_1
Fitting CAT for C2X-Complex_rhown_3x3_depth_lt_1
Fitting EN for C2X-Complex_rhown_3x3_depth_lt_1
Fitting XGB for TOA_9x9_depth_lt_1
Fitting LBM for TOA_9x9_depth_lt_1
Fitting MLP for TOA_9x9_depth_lt_1
Fitting SVR for TOA_9x9_depth_lt_1
Fitting KNN for TOA_9x9_depth_lt_1
Fitting LR f

In [329]:
rows = []

for df_name, model_scores in results.items():
    row = {}
    for model_name, metrics in model_scores.items():
        for metric_name, values in metrics.items():
            if isinstance(values, list):  # Solo para los que tienen listas (folds)
                mean_val = np.mean(values)
                std_val = np.std(values)
                row[(metric_name, model_name)] = f"{mean_val:.2f} ± {std_val:.2f}"
            else:
                # Para el ensemble que tiene un único valor
                row[(metric_name, model_name)] = f"{values:.2f}"
    rows.append((df_name, row))

df_results = pd.DataFrame.from_dict(dict(rows), orient="index")
df_results.columns = pd.MultiIndex.from_tuples(df_results.columns, names=["Metric", "Model"])
df_results = df_results.sort_index(axis=1, level=0)
df_results = df_results.sort_index(axis=0)


In [330]:
df_results

Metric                                     R2                         \
Model                                     CAT           EN  Ensemble   
C2RCC_rhow_5x5_depth_lt_1         0.67 ± 0.13  0.48 ± 0.07      0.61   
C2X-Complex_rhown_3x3_depth_lt_1  0.66 ± 0.12  0.50 ± 0.06  -6049.03   
TOA_9x9_depth_lt_1                0.78 ± 0.12  0.19 ± 0.13      0.46   

Metric                                                                    \
Model                                     KNN          LBM            LR   
C2RCC_rhow_5x5_depth_lt_1         0.74 ± 0.10  0.61 ± 0.24   0.31 ± 0.52   
C2X-Complex_rhown_3x3_depth_lt_1  0.59 ± 0.12  0.55 ± 0.17  -1.12 ± 3.18   
TOA_9x9_depth_lt_1                0.76 ± 0.13  0.70 ± 0.09   0.18 ± 0.36   

Metric                                                                   \
Model                                     MLP           RF          SVR   
C2RCC_rhow_5x5_depth_lt_1         0.71 ± 0.12  0.63 ± 0.18  0.66 ± 0.15   
C2X-Complex_rhown_3x3_depth_lt_1  0.45 ± 0.08  0.63 ± 0.11  0.53 ± 0.09   
TOA_9x9_depth_lt_1                0.65 ± 0.14  0.70 ± 0.13  0.50 ± 0.04   

Metric                                                RMSE               \
Model                                     XGB          CAT           EN   
C2RCC_rhow_5x5_depth_lt_1         0.65 ± 0.19  2.04 ± 0.37  2.64 ± 0.20   
C2X-Complex_rhown_3x3_depth_lt_1  0.61 ± 0.17  2.08 ± 0.17  2.63 ± 0.41   
TOA_9x9_depth_lt_1                0.72 ± 0.12  1.80 ± 0.49  3.58 ± 0.15   

Metric                                                               \
Model                            Ensemble          KNN          LBM   
C2RCC_rhow_5x5_depth_lt_1            2.97  1.81 ± 0.21  2.16 ± 0.50   
C2X-Complex_rhown_3x3_depth_lt_1   370.82  2.30 ± 0.17  2.41 ± 0.25   
TOA_9x9_depth_lt_1                   2.59  1.87 ± 0.37  2.15 ± 0.32   

Metric                                                                   \
Model                                      LR          MLP           RF   
C2RCC_rhow_5x5_depth_lt_1         2.83 ± 0.72  1.93 ± 0.27  2.14 ± 0.35   
C2X-Complex_rhown_3x3_depth_lt_1  4.08 ± 2.64  2.77 ± 0.44  2.21 ± 0.17   
TOA_9x9_depth_lt_1                3.49 ± 0.48  2.31 ± 0.28  2.12 ± 0.39   

Metric                                                      
Model                                     SVR          XGB  
C2RCC_rhow_5x5_depth_lt_1         2.08 ± 0.31  2.08 ± 0.44  
C2X-Complex_rhown_3x3_depth_lt_1  2.52 ± 0.31  2.21 ± 0.28  
TOA_9x9_depth_lt_1                2.84 ± 0.26  2.05 ± 0.44

In [316]:
df_results

Metric                              R2                                         \
Model                              CAT    EN Ensemble   KNN   LBM    LR   MLP   
C2RCC_rhow_1x1_depth_lt_1         0.56  0.45     0.64  0.66  0.62  0.64  0.09   
C2RCC_rhow_3x3_depth_lt_1         0.59  0.40     0.71  0.65  0.54  0.47  0.64   
C2RCC_rhow_5x5_depth_lt_1         0.62  0.49     0.75  0.68  0.68  0.64  0.71   
C2RCC_rhow_9x9_depth_lt_1         0.77  0.48     0.64  0.65  0.78  0.49  0.67   
C2RCC_rhown_1x1_depth_lt_1        0.51  0.42     0.68  0.69  0.56  0.44  0.16   
C2RCC_rhown_3x3_depth_lt_1        0.56  0.39     0.72  0.64  0.53  0.48  0.65   
C2RCC_rhown_5x5_depth_lt_1        0.63  0.47     0.74  0.70  0.73  0.62  0.71   
C2RCC_rhown_9x9_depth_lt_1        0.82  0.47     0.55  0.65  0.80  0.55  0.65   
C2X-Complex_rhow_1x1_depth_lt_1   0.70  0.50     0.60  0.69  0.59  0.15  0.51   
C2X-Complex_rhow_3x3_depth_lt_1   0.73  0.39     0.54  0.61  0.60  0.10  0.67   
C2X-Complex_rhow_5x5_depth_lt_1   0.72  0.37     0.63  0.60  0.65 -2.23  0.55   
C2X-Complex_rhow_9x9_depth_lt_1   0.81  0.53     0.69  0.66  0.73  0.41  0.71   
C2X-Complex_rhown_1x1_depth_lt_1  0.73  0.54     0.70  0.70  0.65  0.55  0.65   
C2X-Complex_rhown_3x3_depth_lt_1  0.67  0.38     0.68  0.56  0.51  0.54  0.61   
C2X-Complex_rhown_5x5_depth_lt_1  0.78  0.44     0.76  0.63  0.72  0.49  0.65   
C2X-Complex_rhown_9x9_depth_lt_1  0.79  0.58     0.76  0.67  0.75  0.45  0.62   
C2X_rhow_1x1_depth_lt_1           0.68  0.63     0.68  0.62  0.54  0.08  0.57   
C2X_rhow_3x3_depth_lt_1           0.65  0.48     0.30  0.61  0.50 -0.88  0.56   
C2X_rhow_5x5_depth_lt_1           0.54  0.57     0.68  0.56  0.36  0.54  0.58   
C2X_rhow_9x9_depth_lt_1           0.68  0.72     0.74  0.74  0.60  0.37  0.58   
C2X_rhown_1x1_depth_lt_1          0.59  0.58     0.58  0.52  0.49  0.33  0.47   
C2X_rhown_3x3_depth_lt_1          0.62  0.33     0.63  0.51  0.49  0.03  0.58   
C2X_rhown_5x5_depth_lt_1          0.51  0.40     0.42  0.46  0.35  0.72  0.53   
C2X_rhown_9x9_depth_lt_1          0.68  0.66     0.66  0.58  0.50  0.47  0.65   
TOA_1x1_depth_lt_1                0.37  0.08     0.40  0.43  0.59  0.27  0.64   
TOA_3x3_depth_lt_1                0.58  0.15     0.48  0.47  0.65  0.45  0.57   
TOA_5x5_depth_lt_1                0.56  0.22     0.48  0.50  0.60  0.38  0.55   
TOA_9x9_depth_lt_1                0.60  0.21     0.43  0.51  0.58 -0.03  0.56   

Metric                                              RMSE                       \
Model                               RF   SVR   XGB   CAT    EN Ensemble   KNN   
C2RCC_rhow_1x1_depth_lt_1         0.65  0.64  0.50  3.17  3.55     2.86  2.76   
C2RCC_rhow_3x3_depth_lt_1         0.62  0.63  0.57  3.04  3.71     2.57  2.82   
C2RCC_rhow_5x5_depth_lt_1         0.67  0.68  0.73  2.94  3.42     2.40  2.68   
C2RCC_rhow_9x9_depth_lt_1         0.72  0.62  0.79  2.27  3.44     2.88  2.81   
C2RCC_rhown_1x1_depth_lt_1        0.61  0.62  0.52  3.34  3.62     2.69  2.64   
C2RCC_rhown_3x3_depth_lt_1        0.59  0.64  0.67  3.17  3.71     2.50  2.87   
C2RCC_rhown_5x5_depth_lt_1        0.67  0.67  0.72  2.88  3.46     2.45  2.62   
C2RCC_rhown_9x9_depth_lt_1        0.73  0.63  0.80  2.02  3.47     3.20  2.81   
C2X-Complex_rhow_1x1_depth_lt_1   0.62  0.58  0.62  2.61  3.36     3.03  2.66   
C2X-Complex_rhow_3x3_depth_lt_1   0.61  0.71  0.62  2.47  3.74     3.24  2.98   
C2X-Complex_rhow_5x5_depth_lt_1   0.59  0.73  0.62  2.51  3.77     2.91  3.02   
C2X-Complex_rhow_9x9_depth_lt_1   0.70  0.66  0.67  2.08  3.25     2.66  2.79   
C2X-Complex_rhown_1x1_depth_lt_1  0.59  0.62  0.64  2.49  3.22     2.62  2.59   
C2X-Complex_rhown_3x3_depth_lt_1  0.57  0.72  0.66  2.72  3.77     2.68  3.16   
C2X-Complex_rhown_5x5_depth_lt_1  0.65  0.71  0.69  2.24  3.57     2.33  2.89   
C2X-Complex_rhown_9x9_depth_lt_1  0.68  0.74  0.70  2.20  3.10     2.35  2.73   
C2X_rhow_1x1_depth_lt_1           0.62  0.68  0.66  2.71  2.90     2.68  2.94   
C2X_rhow_3x3_depth_lt_1       

### Entrenamiento con CV de todo

In [59]:
from itertools import islice

In [78]:
FOLDS = 5

results = {}

for nombre_df, df in list(dfs.items()):
#for nombre_df, df in islice(dfs.items(), 3):
    print(f"\n=== Procesando {nombre_df} ===")
    # Ignoramos las columnas de Date, Lat, Lon y Buoy
    df = df.iloc[:, 4:]

    # Separamos el conjunto de datos en train y test: Train 75% Test 25%
    train, test = train_test_split(df, test_size=0.25, random_state=42, stratify=df["High_Chl"])
    # Seleccionamos la columna que queremos predecir
    target = "Chl"

    # Quitamos esa columna y el indicador de clorofila alta
    X = train.drop(columns=[target, "High_Chl"])
    # Para y cogemos solamente Chl
    y = train[target]
    # Para poder hacer StratifiedKFold y tener el mismo número de valores de Chl alta en cada fold
    y_class = train["High_Chl"]

    # Definimos X e y para test
    X_test = test.drop(columns=[target, "High_Chl"])
    y_test = test[target]

    # Dicts para guardar las predicciones sobre los conjuntos de validación, las y's correspondientes y los índices que corresponden dentro del loop de folds para el ensemble
    val_preds = {name: np.zeros(len(train)) for name in models}
    y_vals = defaultdict(list)
    val_indices = {}
    # Dict para guardar las predicciones sobre test
    test_preds = {name: np.zeros(len(test)) for name in models}

    # scaler_X = RobustScaler()
    # scaler_y = RobustScaler()
    # X_scaled = scaler_X.fit_transform(X)
    # X_test_scaled = scaler_X.transform(X_test)
    # y_scaled = scaler_y.fit_transform(y.values.reshape(-1, 1)).ravel()

    # Stratified KFold de 5 folds
    skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)

    # Dict para guardar resultados
    results[nombre_df] = {name: {'RMSE': [], 'R2': []} for name in models}

    # Loop para entrenar cada uno de los modelos
    for name, model in models.items():
        print(f"\n=== Training {name} ===")
        # Loop de folds, manteniendo la proporción de clases (Chl > 5) con y_class
        for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_class)):
            print(f"Fold {fold+1}")
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

            # Para modelos basados en distancias escalamos los datos
            if name in ["MLP", "SVR", "KNN", "LR", "EN"]:
                # Escalado dentro del loop de folds para evitar data leakage entre folds
                scaler_X = RobustScaler()
                scaler_y = RobustScaler()
                X_train_scaled = scaler_X.fit_transform(X_train)
                X_val_scaled = scaler_X.transform(X_val)
                X_test_scaled = scaler_X.transform(X_test)
                y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
                # Entrenamos modelo con datos escalados
                model.fit(X_train_scaled, y_train_scaled)
                # Predicción sobre val y test, haciendo la transformada inversa para devolver y a su escala
                val_pred = scaler_y.inverse_transform(model.predict(X_val_scaled).reshape(-1, 1)).ravel()
                test_pred = scaler_y.inverse_transform(model.predict(X_test_scaled).reshape(-1, 1)).ravel()
            # Para modelos basados en árboles no es necesario escalar
            else:
                # Entrenamos el modelo
                model.fit(X_train, y_train)
                # Predicción sobre val y test
                val_pred = model.predict(X_val)
                test_pred = model.predict(X_test)

            # Guardamos las predicciones sobre val, las y's que les corresponden y los índices
            val_preds[name][val_idx] = val_pred
            if name == list(models.keys())[0]:
                # Solo lo guardamos una vez
                y_vals[fold] = y_val
                val_indices[fold] = val_idx  # val_idx es un array de índices relativos a train

            # Guardamos la predicción de test, haciendo la media entre los folds
            test_preds[name] += test_pred / FOLDS

            # Calculamos y guardamos métricas
            rmse = np.sqrt(mean_squared_error(y_val, val_pred))
            r2 = r2_score(y_val, val_pred)
            results[nombre_df][name]['RMSE'].append(rmse)
            results[nombre_df][name]['R2'].append(r2)

    # Extendemos el dict de resultados con el ensemble
    results[nombre_df]["Ensemble"] = {'RMSE': [], 'R2': []}

    for fold in range(FOLDS):
        # Índices y valores del fold actual
        fold_val_idx = val_indices[fold]
        meta_X_val = np.vstack([val_preds[model][fold_val_idx] for model in models]).T
        meta_y_val = y_vals[fold]

        # Índices de entrenamiento: todos menos el fold actual
        train_folds = [i for i in range(FOLDS) if i != fold]
        train_idx = np.concatenate([val_indices[i] for i in train_folds])
        meta_X_train = np.vstack([val_preds[model][train_idx] for model in models]).T
        meta_y_train = y.iloc[train_idx]

        # Entrenamos el meta-modelo solo con los otros 4 folds
        meta_model = Ridge().fit(meta_X_train, meta_y_train)

        # Predicción en el fold actual (no visto)
        ensemble_pred = meta_model.predict(meta_X_val)

        rmse = np.sqrt(mean_squared_error(meta_y_val, ensemble_pred))
        r2 = r2_score(meta_y_val, ensemble_pred)
        results[nombre_df]["Ensemble"]['RMSE'].append(rmse)
        results[nombre_df]["Ensemble"]['R2'].append(r2)



# === Evaluación final sobre test ===
    for name in models:
        rmse_test = np.sqrt(mean_squared_error(y_test, test_preds[name]))
        r2_test = r2_score(y_test, test_preds[name])
        results[nombre_df][name]["RMSE test"] = rmse_test
        results[nombre_df][name]["R2 test"] = r2_test

    # Construcción del meta-modelo sobre todo el conjunto de validación
    final_meta_X = np.vstack([val_preds[model] for model in models]).T
    final_meta_y = y.values
    ensemble_model = Ridge().fit(final_meta_X, final_meta_y)

    # Predicción sobre test del ensemble
    meta_X_test = np.vstack([test_preds[model] for model in models]).T
    ensemble_test_pred = ensemble_model.predict(meta_X_test)
    # Evaluación del ensemble sobre test
    rmse_ens_test = np.sqrt(mean_squared_error(y_test, ensemble_test_pred))
    r2_ens_test = r2_score(y_test, ensemble_test_pred)
    results[nombre_df]["Ensemble"]["RMSE test"] = rmse_ens_test
    results[nombre_df]["Ensemble"]["R2 test"] = r2_ens_test



=== Procesando C2RCC_rhow_5x5_depth_lt_1 ===

=== Training XGB ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LBM ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training MLP ===
Fold 1
Fold 2


/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Fold 3


/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Fold 4
Fold 5

=== Training KNN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training RF ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training CAT ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training EN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Procesando C2X-Complex_rhown_3x3_depth_lt_1 ===

=== Training XGB ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LBM ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training MLP ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training KNN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training RF ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training CAT ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training EN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Procesando TOA_9x9_depth_lt_1 ===

=== Training XGB ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LBM ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training MLP ===
Fold 1


/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Fold 2
Fold 3


/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Fold 4
Fold 5

=== Training KNN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training RF ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training CAT ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training EN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


In [26]:
with open(f"training_results/results_entrenamiento_CV_{depth}.pkl", "wb") as f:
    pickle.dump(results, f)

In [38]:
with open(f"training_results/results_entrenamiento_CV_{depth}.pkl", "rb") as f:
    results = pickle.load(f)

In [79]:
rows = []

for df_name, model_scores in results.items():
    row = {}
    for model_name, metrics in model_scores.items():
        for metric_name, values in metrics.items():
            if isinstance(values, list):  # Solo para los que tienen listas (folds)
                mean_val = np.mean(values)
                std_val = np.std(values)
                row[(metric_name, model_name)] = f"{mean_val:.2f} ± {std_val:.2f}"
            else:
                # Para el ensemble que tiene un único valor
                row[(metric_name, model_name)] = f"{values:.2f}"
    rows.append((df_name, row))

df_results = pd.DataFrame.from_dict(dict(rows), orient="index")
df_results.columns = pd.MultiIndex.from_tuples(df_results.columns, names=["Metric", "Model"])
df_results = df_results.sort_index(axis=1, level=0)
df_results = df_results.sort_index(axis=0)

In [80]:
df_results

Metric                                     R2                            \
Model                                     CAT           EN     Ensemble   
C2RCC_rhow_5x5_depth_lt_1         0.69 ± 0.10  0.40 ± 0.05  0.58 ± 0.18   
C2X-Complex_rhown_3x3_depth_lt_1  0.57 ± 0.23  0.37 ± 0.11  0.37 ± 0.30   
TOA_9x9_depth_lt_1                0.77 ± 0.13  0.01 ± 0.01  0.66 ± 0.13   

Metric                                                                   \
Model                                     KNN          LBM          MLP   
C2RCC_rhow_5x5_depth_lt_1         0.67 ± 0.13  0.60 ± 0.20  0.53 ± 0.12   
C2X-Complex_rhown_3x3_depth_lt_1  0.49 ± 0.24  0.42 ± 0.34  0.33 ± 0.17   
TOA_9x9_depth_lt_1                0.72 ± 0.10  0.66 ± 0.12  0.48 ± 0.19   

Metric                                                     R2 test        ...  \
Model                                      RF          XGB     CAT    EN  ...   
C2RCC_rhow_5x5_depth_lt_1         0.64 ± 0.09  0.65 ± 0.09    0.76  0.37  ...   
C2X-Complex_rhown_3x3_depth_lt_1  0.55 ± 0.24  0.52 ± 0.31    0.69  0.38  ...   
TOA_9x9_depth_lt_1                0.69 ± 0.13  0.66 ± 0.20    0.70  0.09  ...   

Metric                                   RMSE              RMSE test        \
Model                                      RF          XGB       CAT    EN   
C2RCC_rhow_5x5_depth_lt_1         2.13 ± 0.11  2.10 ± 0.13      2.29  3.74   
C2X-Complex_rhown_3x3_depth_lt_1  2.31 ± 0.46  2.37 ± 0.59      2.63  3.74   
TOA_9x9_depth_lt_1                2.15 ± 0.54  2.20 ± 0.69      2.15  3.71   

Metric                                                                   
Model                            Ensemble   KNN   LBM   MLP    RF   XGB  
C2RCC_rhow_5x5_depth_lt_1            2.29  2.47  2.45  2.66  2.52  2.32  
C2X-Complex_rhown_3x3_depth_lt_1     2.79  2.93  2.88  3.51  2.95  2.79  
TOA_9x9_depth_lt_1                   2.14  2.45  2.63  2.55  2.51  2.58  

[3 rows x 32 columns]

In [83]:
# Asumiendo que tu DataFrame se llama df_results
df_sorted = df_results["R2 test"].copy()

# Añadir una columna auxiliar con el R2 máximo por fila
df_sorted["max_R2"] = df_sorted.max(axis=1)

# Ordenar por esa columna en orden descendente
df_sorted = df_sorted.sort_values("max_R2", ascending=False)

# Eliminar la columna auxiliar
df_sorted = df_sorted.drop(columns="max_R2")



In [82]:
df_sorted

Model,CAT,EN,Ensemble,KNN,LBM,MLP,RF,XGB
TOA_9x9_depth_lt_1,0.77 ± 0.13,0.01 ± 0.01,0.66 ± 0.13,0.72 ± 0.10,0.66 ± 0.12,0.48 ± 0.19,0.69 ± 0.13,0.66 ± 0.20
C2RCC_rhow_5x5_depth_lt_1,0.69 ± 0.10,0.40 ± 0.05,0.58 ± 0.18,0.67 ± 0.13,0.60 ± 0.20,0.53 ± 0.12,0.64 ± 0.09,0.65 ± 0.09
C2X-Complex_rhown_3x3_depth_lt_1,0.57 ± 0.23,0.37 ± 0.11,0.37 ± 0.30,0.49 ± 0.24,0.42 ± 0.34,0.33 ± 0.17,0.55 ± 0.24,0.52 ± 0.31


In [84]:
df_sorted

Model,CAT,EN,Ensemble,KNN,LBM,MLP,RF,XGB
C2RCC_rhow_5x5_depth_lt_1,0.76,0.37,0.77,0.73,0.73,0.68,0.72,0.76
TOA_9x9_depth_lt_1,0.70,0.09,0.70,0.60,0.54,0.57,0.58,0.56
C2X-Complex_rhown_3x3_depth_lt_1,0.69,0.38,0.65,0.62,0.63,0.45,0.61,0.65


In [40]:
df_results["R2 test"]

Model,CAT,EN,Ensemble,KNN,LBM,LR,MLP,RF,SVR,XGB
C2RCC_rhow_1x1_depth_lt_1,0.74,0.46,0.74,0.72,0.70,0.62,0.61,0.69,0.69,0.75
C2RCC_rhow_3x3_depth_lt_1,0.71,0.42,0.66,0.66,0.72,0.39,0.56,0.67,0.72,0.71
C2RCC_rhow_5x5_depth_lt_1,0.76,0.49,0.76,0.73,0.76,0.39,0.65,0.71,0.77,0.78
C2RCC_rhow_9x9_depth_lt_1,0.81,0.48,0.76,0.70,0.77,0.63,0.69,0.72,0.75,0.80
C2RCC_rhown_1x1_depth_lt_1,0.73,0.45,0.77,0.76,0.69,0.53,0.62,0.68,0.68,0.75
C2RCC_rhown_3x3_depth_lt_1,0.74,0.42,0.64,0.63,0.71,0.43,0.67,0.65,0.71,0.72
C2RCC_rhown_5x5_depth_lt_1,0.76,0.48,0.71,0.69,0.76,0.56,0.67,0.72,0.75,0.79
C2RCC_rhown_9x9_depth_lt_1,0.79,0.48,0.72,0.68,0.77,0.53,0.63,0.73,0.75,0.77
C2X-Complex_rhow_1x1_depth_lt_1,0.73,0.53,0.78,0.75,0.68,0.43,0.62,0.64,0.62,0.75
C2X-Complex_rhow_3x3_depth_lt_1,0.72,0.39,0.74,0.68,0.64,0.11,0.61,0.64,0.73,0.66


In [29]:
df_results["R2"]

Model,CAT,EN,Ensemble,KNN,LBM,LR,MLP,RF,SVR,XGB
C2RCC_rhow_1x1_depth_gt_3,0.39 ± 0.10,0.25 ± 0.06,0.55 ± 0.09,0.35 ± 0.12,0.38 ± 0.11,-0.54 ± 1.52,0.39 ± 0.08,0.35 ± 0.10,0.37 ± 0.11,0.37 ± 0.08
C2RCC_rhow_3x3_depth_gt_3,0.40 ± 0.13,0.27 ± 0.05,0.61 ± 0.07,0.39 ± 0.10,0.40 ± 0.10,0.24 ± 0.18,0.35 ± 0.08,0.42 ± 0.13,0.40 ± 0.10,0.38 ± 0.15
C2RCC_rhow_5x5_depth_gt_3,0.48 ± 0.13,0.29 ± 0.05,0.65 ± 0.10,0.46 ± 0.11,0.45 ± 0.13,0.08 ± 0.58,0.40 ± 0.05,0.46 ± 0.13,0.48 ± 0.09,0.47 ± 0.14
C2RCC_rhow_9x9_depth_gt_3,0.51 ± 0.08,0.29 ± 0.05,0.68 ± 0.09,0.42 ± 0.12,0.50 ± 0.09,0.30 ± 0.17,0.39 ± 0.05,0.48 ± 0.08,0.54 ± 0.06,0.49 ± 0.09
C2RCC_rhown_1x1_depth_gt_3,0.39 ± 0.09,0.25 ± 0.06,0.58 ± 0.08,0.35 ± 0.10,0.34 ± 0.11,0.23 ± 0.08,0.32 ± 0.08,0.38 ± 0.11,0.38 ± 0.11,0.42 ± 0.08
C2RCC_rhown_3x3_depth_gt_3,0.40 ± 0.11,0.27 ± 0.05,0.59 ± 0.12,0.40 ± 0.09,0.44 ± 0.13,0.22 ± 0.14,0.42 ± 0.09,0.43 ± 0.10,0.42 ± 0.10,0.39 ± 0.14
C2RCC_rhown_5x5_depth_gt_3,0.46 ± 0.11,0.29 ± 0.05,0.70 ± 0.06,0.50 ± 0.11,0.53 ± 0.09,0.20 ± 0.22,0.47 ± 0.05,0.49 ± 0.10,0.49 ± 0.09,0.50 ± 0.09
C2RCC_rhown_9x9_depth_gt_3,0.52 ± 0.10,0.29 ± 0.05,0.70 ± 0.06,0.45 ± 0.13,0.48 ± 0.12,0.05 ± 0.47,0.38 ± 0.07,0.51 ± 0.09,0.56 ± 0.06,0.50 ± 0.11
C2X-Complex_rhow_1x1_depth_gt_3,0.45 ± 0.08,0.19 ± 0.19,0.66 ± 0.04,0.34 ± 0.07,0.33 ± 0.13,-1.95 ± 3.97,0.36 ± 0.07,0.39 ± 0.10,0.41 ± 0.07,0.45 ± 0.08
C2X-Complex_rhow_3x3_depth_gt_3,0.48 ± 0.05,0.28 ± 0.02,0.71 ± 0.04,0.45 ± 0.07,0.34 ± 0.07,0.29 ± 0.16,0.47 ± 0.05,0.44 ± 0.08,0.46 ± 0.12,0.39 ± 0.12
